In [ ]:
!pip -q install fastapi uvicorn sqlalchemy pydantic nest_asyncio

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.7.3) ...
Setting up cloudflared (2026.7.3) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
import nest_asyncio
import threading
import uvicorn

from fastapi import FastAPI

nest_asyncio.apply()

In [ ]:
app = FastAPI(
    title="Blinkit Backend",
    version="1.0"
)

@app.get("/")
def home():
    return {
        "message":"Welcome to Blinkit Backend"
    }

@app.get("/health")
def health():
    return {
        "status":"Running"
    }

In [ ]:
def run():
    uvicorn.run(app, host="0.0.0.0", port=3000)

thread = threading.Thread(target=run)

thread.start()

In [ ]:
import subprocess
import time
import re

# Start Cloudflare Tunnel
process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:3000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url = None

print("Creating Cloudflare Tunnel...\n")

while True:
    line = process.stdout.readline()

    if line:
        print(line.strip())

        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)

        if match:
            url = match.group(0)
            break

print("\nTunnel Created Successfully!")
print("Swagger URL:")
print(url + "/docs")

Creating Cloudflare Tunnel...

2026-07-29T03:42:47Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-29T03:42:47Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-29T03:42:54Z INF +--------------------------------------------------------------------------------------------+
2026-07-29T03:42:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-29T03:42:54Z INF |  https://proven-toward-

In [ ]:
from sqlalchemy import (
    create_engine,
    Column,
    Integer,
    String,
    Float,
    ForeignKey
)

from sqlalchemy.orm import (
    declarative_base,
    relationship,
    sessionmaker
)

In [ ]:
DATABASE_URL = "sqlite:///blinkit.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()

In [ ]:
def get_db():

    db = SessionLocal()

    try:
        yield db

    finally:
        db.close()

In [ ]:
class User(Base):

    __tablename__ = "users"

    id = Column(Integer, primary_key=True, index=True)

    username = Column(String, unique=True, nullable=False)

    password = Column(String, nullable=False)

    cart_items = relationship("Cart", back_populates="user")

    orders = relationship("Order", back_populates="user")

In [ ]:
class Category(Base):

    __tablename__ = "categories"

    id = Column(Integer, primary_key=True)

    name = Column(String, unique=True)

    products = relationship(
        "Product",
        back_populates="category"
    )

In [ ]:
class Product(Base):

    __tablename__ = "products"

    id = Column(Integer, primary_key=True)

    name = Column(String)

    category_id = Column(
        Integer,
        ForeignKey("categories.id")
    )

    price = Column(Float)

    weight = Column(Float)

    stock = Column(Integer)

    category = relationship(
        "Category",
        back_populates="products"
    )

    cart_items = relationship(
        "Cart",
        back_populates="product"
    )

In [ ]:
class Cart(Base):

    __tablename__ = "cart"

    id = Column(Integer, primary_key=True)

    user_id = Column(
        Integer,
        ForeignKey("users.id")
    )

    product_id = Column(
        Integer,
        ForeignKey("products.id")
    )

    quantity = Column(Integer)

    user = relationship(
        "User",
        back_populates="cart_items"
    )

    product = relationship(
        "Product",
        back_populates="cart_items"
    )

In [ ]:
class Order(Base):

    __tablename__ = "orders"

    id = Column(Integer, primary_key=True)

    user_id = Column(
        Integer,
        ForeignKey("users.id")
    )

    total_amount = Column(Float)

    payment_method = Column(String)

    bag = Column(String)

    user = relationship(
        "User",
        back_populates="orders"
    )

In [ ]:
Base.metadata.create_all(bind=engine)

print("All Tables Created Successfully")

All Tables Created Successfully


In [ ]:
db = SessionLocal()

print("Database Connected")

Database Connected


In [ ]:
from sqlalchemy import inspect

inspector = inspect(engine)

print(inspector.get_table_names())

['cart', 'categories', 'orders', 'products', 'users']


In [ ]:
if db.query(Category).count() == 0:

    categories = [

        Category(name="Fruits"),

        Category(name="Vegetables"),

        Category(name="Dairy"),

        Category(name="Snacks"),

        Category(name="Beverages")

    ]

    db.add_all(categories)

    db.commit()

    print("Categories Added")

else:

    print("Categories Already Exist")

Categories Added


In [ ]:
if db.query(Product).count() == 0:

    products = [

        Product(name="Apple", category_id=1, price=120, weight=1, stock=100),

        Product(name="Banana", category_id=1, price=60, weight=1, stock=150),

        Product(name="Tomato", category_id=2, price=40, weight=1, stock=200),

        Product(name="Potato", category_id=2, price=30, weight=1, stock=250),

        Product(name="Milk", category_id=3, price=55, weight=1, stock=80),

        Product(name="Curd", category_id=3, price=45, weight=1, stock=70),

        Product(name="Chips", category_id=4, price=20, weight=1, stock=120),

        Product(name="Biscuits", category_id=4, price=25, weight=1, stock=100),

        Product(name="Coke", category_id=5, price=40, weight=1, stock=90),

        Product(name="Juice", category_id=5, price=80, weight=1, stock=60)

    ]

    db.add_all(products)

    db.commit()

    print("Products Added")

else:

    print("Products Already Exist")

Products Added


In [ ]:
products = db.query(Product).all()

for product in products:

    print(

        product.id,

        product.name,

        product.price,

        product.stock

    )

1 Apple 120.0 100
2 Banana 60.0 150
3 Tomato 40.0 200
4 Potato 30.0 250
5 Milk 55.0 80
6 Curd 45.0 70
7 Chips 20.0 120
8 Biscuits 25.0 100
9 Coke 40.0 90
10 Juice 80.0 60


In [ ]:
from pydantic import BaseModel

In [ ]:
class UserRegister(BaseModel):
    username: str
    password: str


class UserLogin(BaseModel):
    username: str
    password: str

In [ ]:
@app.post("/register")
def register(user: UserRegister):

    db = SessionLocal()

    existing = db.query(User).filter(
        User.username == user.username
    ).first()

    if existing:
        db.close()
        return {
            "success": False,
            "message": "Username already exists"
        }

    new_user = User(
        username=user.username,
        password=user.password
    )

    db.add(new_user)
    db.commit()
    db.refresh(new_user)

    result = {
        "success": True,
        "message": "Registration Successful",
        "user_id": new_user.id
    }

    db.close()

    return result

In [ ]:
@app.post("/login")
def login(user: UserLogin):

    db = SessionLocal()

    existing = db.query(User).filter(
        User.username == user.username,
        User.password == user.password
    ).first()

    db.close()

    if existing:

        return {
            "success": True,
            "message": "Login Successful",
            "user_id": existing.id
        }

    return {
        "success": False,
        "message": "Invalid Username or Password"
    }

In [ ]:
@app.get("/categories")
def get_categories():

    db = SessionLocal()

    categories = db.query(Category).all()

    data = []

    for category in categories:

        data.append({

            "id": category.id,

            "name": category.name

        })

    db.close()

    return data

In [ ]:
@app.get("/products")
def get_products():

    db = SessionLocal()

    products = db.query(Product).all()

    result = []

    for product in products:

        result.append({

            "id": product.id,

            "name": product.name,

            "category": product.category.name,

            "price": product.price,

            "weight": product.weight,

            "stock": product.stock

        })

    db.close()

    return result

In [ ]:
@app.get("/products/{category_id}")
def products_by_category(category_id: int):

    db = SessionLocal()

    products = db.query(Product).filter(
        Product.category_id == category_id
    ).all()

    result = []

    for product in products:

        result.append({

            "id": product.id,

            "name": product.name,

            "price": product.price,

            "weight": product.weight,

            "stock": product.stock

        })

    db.close()

    return result

In [ ]:
class CartRequest(BaseModel):
    user_id: int
    product_id: int
    quantity: int

In [ ]:
@app.post("/cart/add")
def add_to_cart(cart: CartRequest):

    db = SessionLocal()

    user = db.query(User).filter(User.id == cart.user_id).first()

    if not user:
        db.close()
        return {
            "success": False,
            "message": "User not found"
        }

    product = db.query(Product).filter(Product.id == cart.product_id).first()

    if not product:
        db.close()
        return {
            "success": False,
            "message": "Product not found"
        }

    if product.stock < cart.quantity:
        db.close()
        return {
            "success": False,
            "message": "Insufficient stock"
        }

    existing = db.query(Cart).filter(
        Cart.user_id == cart.user_id,
        Cart.product_id == cart.product_id
    ).first()

    if existing:
        existing.quantity += cart.quantity
    else:
        item = Cart(
            user_id=cart.user_id,
            product_id=cart.product_id,
            quantity=cart.quantity
        )
        db.add(item)

    db.commit()
    db.close()

    return {
        "success": True,
        "message": "Product added to cart"
    }

In [ ]:
@app.get("/cart/{user_id}")
def view_cart(user_id: int):

    db = SessionLocal()

    cart_items = db.query(Cart).filter(
        Cart.user_id == user_id
    ).all()

    result = []

    total = 0

    for item in cart_items:

        product = db.query(Product).filter(
            Product.id == item.product_id
        ).first()

        amount = product.price * item.quantity

        total += amount

        result.append({
            "product_id": product.id,
            "product_name": product.name,
            "price": product.price,
            "quantity": item.quantity,
            "amount": amount
        })

    db.close()

    return {
        "cart": result,
        "total": total
    }

In [ ]:
class RemoveCart(BaseModel):
    user_id: int
    product_id: int

In [ ]:
@app.delete("/cart/remove")
def remove_from_cart(data: RemoveCart):

    db = SessionLocal()

    item = db.query(Cart).filter(
        Cart.user_id == data.user_id,
        Cart.product_id == data.product_id
    ).first()

    if not item:
        db.close()
        return {
            "success": False,
            "message": "Product not found in cart"
        }

    db.delete(item)

    db.commit()

    db.close()

    return {
        "success": True,
        "message": "Product removed successfully"
    }

In [ ]:
class BillRequest(BaseModel):
    user_id: int
    need_bag: bool

In [ ]:
def calculate_bill(user_id, need_bag):

    db = SessionLocal()

    cart_items = db.query(Cart).filter(
        Cart.user_id == user_id
    ).all()

    if len(cart_items) == 0:
        db.close()
        return None

    products = []

    total_amount = 0
    total_weight = 0

    for item in cart_items:

        product = db.query(Product).filter(
            Product.id == item.product_id
        ).first()

        amount = product.price * item.quantity
        weight = product.weight * item.quantity

        total_amount += amount
        total_weight += weight

        products.append({
            "product": product.name,
            "quantity": item.quantity,
            "price": product.price,
            "amount": amount
        })

    discount = 0

    if total_weight >= 10:
        discount = total_amount * 0.20

    elif total_weight >= 5:
        discount = total_amount * 0.15

    bag_charge = 10 if need_bag else 0

    final_amount = total_amount - discount + bag_charge

    db.close()

    return {
        "products": products,
        "total_weight": total_weight,
        "total_amount": total_amount,
        "discount": discount,
        "bag_charge": bag_charge,
        "final_amount": final_amount
    }

In [ ]:
@app.post("/bill")
def generate_bill(request: BillRequest):

    bill = calculate_bill(
        request.user_id,
        request.need_bag
    )

    if bill is None:
        return {
            "success": False,
            "message": "Cart is Empty"
        }

    return bill

In [ ]:
class PaymentRequest(BaseModel):

    user_id: int

    need_bag: bool

    payment_method: str

In [ ]:
@app.post("/payment")
def payment(request: PaymentRequest):

    db = SessionLocal()

    bill = calculate_bill(
        request.user_id,
        request.need_bag
    )

    if bill is None:
        db.close()
        return {
            "success": False,
            "message": "Cart is Empty"
        }

    methods = ["Cash", "UPI", "Card"]

    if request.payment_method not in methods:
        db.close()
        return {
            "success": False,
            "message": "Invalid Payment Method"
        }

    order = Order(
        user_id=request.user_id,
        total_amount=bill["final_amount"],
        payment_method=request.payment_method,
        bag="Yes" if request.need_bag else "No"
    )

    db.add(order)

    cart_items = db.query(Cart).filter(
        Cart.user_id == request.user_id
    ).all()

    for item in cart_items:

        product = db.query(Product).filter(
            Product.id == item.product_id
        ).first()

        product.stock -= item.quantity

        db.delete(item)

    db.commit()

    order_id = order.id

    db.close()

    return {
        "success": True,
        "message": "Order Placed Successfully",
        "order_id": order_id,
        "payment_method": request.payment_method,
        "amount_paid": bill["final_amount"]
    }

In [ ]:
@app.get("/orders/{user_id}")
def order_history(user_id: int):

    db = SessionLocal()

    orders = db.query(Order).filter(
        Order.user_id == user_id
    ).all()

    result = []

    for order in orders:

        result.append({

            "order_id": order.id,

            "amount": order.total_amount,

            "payment": order.payment_method,

            "bag": order.bag

        })

    db.close()

    return result

In [ ]:
@app.get("/products/search")
def search_products(keyword: str):

    db = SessionLocal()

    products = db.query(Product).filter(
        Product.name.ilike(f"%{keyword}%")
    ).all()

    result = []

    for p in products:

        result.append({

            "id": p.id,

            "name": p.name,

            "category": p.category.name,

            "price": p.price,

            "stock": p.stock

        })

    db.close()

    return result

In [ ]:
@app.get("/products/filter")
def filter_price(max_price: float):

    db = SessionLocal()

    products = db.query(Product).filter(
        Product.price <= max_price
    ).all()

    data=[]

    for p in products:

        data.append({

            "id":p.id,

            "name":p.name,

            "price":p.price,

            "stock":p.stock

        })

    db.close()

    return data

In [ ]:
class UpdateCart(BaseModel):

    user_id:int

    product_id:int

    quantity:int

In [ ]:
@app.put("/cart/update")
def update_cart(item: UpdateCart):

    db = SessionLocal()

    cart = db.query(Cart).filter(

        Cart.user_id==item.user_id,

        Cart.product_id==item.product_id

    ).first()

    if not cart:

        db.close()

        return {

            "success":False,

            "message":"Item not found"

        }

    cart.quantity=item.quantity

    db.commit()

    db.close()

    return {

        "success":True,

        "message":"Quantity Updated"

    }

In [ ]:
@app.get("/user/{user_id}")
def profile(user_id:int):

    db=SessionLocal()

    user=db.query(User).filter(

        User.id==user_id

    ).first()

    if not user:

        db.close()

        return{

            "success":False,

            "message":"User not found"

        }

    data={

        "id":user.id,

        "username":user.username

    }

    db.close()

    return data

In [ ]:
@app.get("/dashboard")
def dashboard():

    db=SessionLocal()

    return{

        "users":db.query(User).count(),

        "categories":db.query(Category).count(),

        "products":db.query(Product).count(),

        "orders":db.query(Order).count(),

        "cart_items":db.query(Cart).count()

    }

In [ ]:
@app.get("/low-stock")
def low_stock():

    db=SessionLocal()

    products=db.query(Product).filter(

        Product.stock<20

    ).all()

    result=[]

    for p in products:

        result.append({

            "id":p.id,

            "name":p.name,

            "stock":p.stock

        })

    db.close()

    return result

In [ ]:
class Restock(BaseModel):

    product_id:int

    quantity:int

In [ ]:
@app.put("/restock")
def restock(data:Restock):

    db=SessionLocal()

    product=db.query(Product).filter(

        Product.id==data.product_id

    ).first()

    if not product:

        db.close()

        return{

            "success":False,

            "message":"Product not found"

        }

    product.stock+=data.quantity

    db.commit()

    db.refresh(product)

    db.close()

    return{

        "success":True,

        "message":"Stock Updated",

        "current_stock":product.stock

    }